# Notebook 2: What is Microsoft Foundry?
## Platform Overview, Key Concepts & Your First API Call

**Source:** [Microsoft Learn — What is Microsoft Foundry?](https://learn.microsoft.com/en-us/azure/foundry/what-is-foundry?tabs=python)

---

This notebook covers:
1. What Microsoft Foundry is and how it evolved
2. Key concepts and resource model
3. Available models (1,900+)
4. Your first API call (Python)
5. Key capabilities — agents, tools, memory, observability
6. SDKs and developer surfaces

## What is Microsoft Foundry?

**Microsoft Foundry** is a unified Azure platform-as-a-service offering for:
- **Enterprise AI operations**
- **Model builders**
- **Application development**

It combines production-grade infrastructure with friendly interfaces, enabling developers to focus on building applications rather than managing infrastructure.

### What it unifies

Microsoft Foundry brings together **agents, models, and tools** under a single management grouping with built-in enterprise-readiness capabilities:

| Capability | Description |
|---|---|
| **Tracing** | End-to-end request tracing across agents and models |
| **Monitoring** | Real-time observability dashboards |
| **Evaluations** | Built-in evaluation frameworks for agent quality |
| **Enterprise Setup** | Customizable configurations for security and governance |
| **Unified RBAC** | Role-based access control across all resources |
| **Networking & Policies** | Centralized under one Azure resource provider namespace |

---
## Evolution of Foundry

Foundry consolidates several previous Azure AI services and tools into a unified platform:

| Dimension | Previous | Current |
|---|---|---|
| **Brand** | Azure AI Studio / Azure AI Foundry | Microsoft Foundry |
| **Brand** | Azure AI Services | Foundry Tools |
| **Portal** | Foundry (classic) | [Foundry portal](https://ai.azure.com) |
| **Agent API** | Assistants API (Agents v0.5/v1) | Responses API (Agents v2) |
| **API versioning** | Monthly `api-version` params | v1 stable routes (`/openai/v1/`) |
| **Resource model** | Hub + Azure OpenAI + Azure AI Services | Foundry resource (single, with projects) |
| **SDKs & endpoints** | Multiple packages (`azure-ai-inference`, `azure-ai-generative`, `azure-ai-ml`, `AzureOpenAI()`) against 5+ endpoints | Unified project client (`azure-ai-projects` 2.x) + `OpenAI()` against one project endpoint |
| **Terminology** | Threads, Messages, Runs, Assistants | Conversations, Items, Responses, Agent Versions |

> **Coming from Azure OpenAI?** You can [upgrade your Azure OpenAI resource to a Foundry resource](https://learn.microsoft.com/en-us/azure/foundry/how-to/upgrade-azure-openai) while preserving your endpoint, API keys, and existing state.

---
## Architecture: The New Resource Model

```
┌──────────────────────────────────────────────────────────────┐
│                   Microsoft Foundry                          │
│                                                              │
│  ┌────────────────────────────────────────────────────────┐  │
│  │              Foundry Resource (single)                 │  │
│  │                                                        │  │
│  │   ┌──────────────┐  ┌──────────────┐  ┌────────────┐  │  │
│  │   │  Project A    │  │  Project B    │  │ Project C   │  │  │
│  │   │              │  │              │  │             │  │  │
│  │   │ • Agents     │  │ • Agents     │  │ • Agents    │  │  │
│  │   │ • Models     │  │ • Models     │  │ • Models    │  │  │
│  │   │ • Tools      │  │ • Tools      │  │ • Tools     │  │  │
│  │   │ • Evals      │  │ • Evals      │  │ • Evals     │  │  │
│  │   └──────────────┘  └──────────────┘  └────────────┘  │  │
│  │                                                        │  │
│  │  Unified: RBAC · Networking · Policies · Monitoring    │  │
│  └────────────────────────────────────────────────────────┘  │
│                                                              │
│  ┌──────────────────────────┐  ┌──────────────────────────┐  │
│  │   Foundry Tools          │  │   Foundry Models         │  │
│  │   (formerly AI Services) │  │   (1,900+ models)        │  │
│  └──────────────────────────┘  └──────────────────────────┘  │
└──────────────────────────────────────────────────────────────┘
```

**Key difference from the old model:**
- **Before:** Hub + Azure OpenAI + Azure AI Services = 3 separate resources to manage
- **Now:** A single **Foundry resource** with **projects** underneath — one endpoint, one SDK, one RBAC surface

---
## Available Models

Foundry gives you access to **over 1,900 models** from Microsoft, OpenAI, Anthropic, Mistral, xAI, Meta, DeepSeek, Hugging Face, and more.

| Model Family | Best For |
|---|---|
| **GPT-5** | Most capable — complex reasoning, multi-step tasks, and multimodal scenarios |
| **GPT-4.1** | Best balance of capability and cost for production workloads |
| **GPT-4.1 mini** | Fastest — low-latency, high-throughput scenarios |
| **Claude** | Advanced reasoning, code generation, and multimodal tasks |
| **Grok** | Reasoning, coding, and data extraction |
| **Mistral** | Code generation, multilingual, and general-purpose chat |
| **DeepSeek-R1** | Open-weight reasoning at scale |
| **Phi-4** | Small language model — on-device or resource-constrained environments |
| **Meta Llama** | Open models — customization and fine-tuning |

> Browse the full catalog in the [Foundry Models overview](https://learn.microsoft.com/en-us/azure/foundry/concepts/foundry-models-overview).

---
## Your First API Call

The unified SDK pattern: create an `AIProjectClient` from your project endpoint, get an OpenAI client from it, and call the Responses API.

**Project endpoint format:** `https://<resource_name>.ai.azure.com/api/projects/<project_name>`

### Install the required packages

In [ ]:
# Install the unified Foundry SDK (azure-ai-projects 2.x) and identity package
!pip install azure-ai-projects azure-identity openai

In [ ]:
# Verify packages are installed
try:
    import azure.ai.projects
    print(f"azure-ai-projects installed successfully")
except ImportError:
    print("ERROR: azure-ai-projects not found — run the install cell above")

try:
    import azure.identity
    print(f"azure-identity installed successfully")
except ImportError:
    print("ERROR: azure-identity not found")

try:
    import openai
    print(f"openai installed successfully (version: {openai.__version__})")
except ImportError:
    print("ERROR: openai not found")

### Configure your project endpoint

Replace the placeholder below with your actual Foundry project endpoint.

You can find this in:
- **Foundry portal** > Your Project > **Overview** page
- Or construct it: `https://<resource_name>.ai.azure.com/api/projects/<project_name>`

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Option 1: Set directly (replace with your actual endpoint)
# PROJECT_ENDPOINT = "https://your_resource_name.ai.azure.com/api/projects/your_project_name"

# Option 2: Load from environment variable
PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT", "your_project_endpoint")

if PROJECT_ENDPOINT == "your_project_endpoint":
    print("WARNING: Please set your PROJECT_ENDPOINT above before running the API call.")
    print("Format: https://<resource_name>.ai.azure.com/api/projects/<project_name>")
else:
    print(f"Project endpoint configured: {PROJECT_ENDPOINT[:50]}...")

### Make your first Responses API call

This is the new unified pattern — `AIProjectClient` + `OpenAI()` against a single project endpoint:

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Create project client — single entry point for all Foundry APIs
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# Get an OpenAI-compatible client from the project
openai = project.get_openai_client()

# Run a Responses API call (the new agent-native API replacing Chat Completions)
response = openai.responses.create(
    model="gpt-4.1-mini",  # Supports all Foundry direct models
    input="What is the size of France in square miles?",
)

print(f"Response: {response.output_text}")

### Understanding the code pattern

```
┌─────────────────────────────────────────────────────┐
│  Your Code                                          │
│                                                     │
│  1. AIProjectClient(endpoint, credential)           │
│     └─── Single entry point to Foundry              │
│                                                     │
│  2. project.get_openai_client()                     │
│     └─── Returns OpenAI-compatible client           │
│          routed through your project endpoint       │
│                                                     │
│  3. openai.responses.create(model, input)           │
│     └─── New Responses API (replaces Completions)   │
│          Supports: text, tools, agents, streaming   │
└─────────────────────────────────────────────────────┘
```

**Why `responses.create` instead of `chat.completions.create`?**
- The Responses API is the new **agent-native** API in Foundry
- It supports built-in tool calling, agent orchestration, and memory
- Chat Completions still works, but Responses API is the recommended path forward

### Alternative: REST API call

You can also call the Foundry API directly via REST:

```bash
curl -X POST https://YOUR-FOUNDRY-RESOURCE-NAME.services.ai.azure.com/api/projects/YOUR-PROJECT-NAME/openai/v1/responses \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $AZURE_AI_AUTH_TOKEN" \
  -d '{
        "model": "gpt-4.1-mini",
        "input": "What is the size of France in square miles?"
      }'
```

> Notice the new stable route: `/openai/v1/responses` — no more monthly `api-version` query parameters!

---
## Key Capabilities

### Build Agents

| Feature | Description |
|---|---|
| **Multi-agent orchestration** | Build collaborative agent behavior and complex workflow execution using SDKs for C# and Python |
| **Tool catalog** | Connect over 1,400 tools through public and private catalogs |
| **Memory** | Retain and recall contextual information across interactions without requiring repeated input |
| **Foundry IQ** | Ground agent responses in enterprise or web content with citation-backed answers |
| **Publishing** | Publish agents to Microsoft 365, Teams, BizChat, or containerized deployments |

### Operate and Govern

| Feature | Description |
|---|---|
| **Real-time observability** | Monitor performance with built-in metrics, model tracking, and continuous evaluation |
| **Centralized AI asset management** | Manage all agents, models, and tools from the Operate section — including agents from other clouds |
| **Enterprise controls** | Full authentication support for MCP and A2A, AI gateway integration, and Azure Policy integration |

---
## Who is Foundry For?

| Audience | What They Do | Start Here |
|---|---|---|
| **Application developers** | Build AI-powered products with agents, models, and tools | [Quickstart](https://learn.microsoft.com/en-us/azure/foundry/quickstarts/get-started-code) |
| **ML engineers & data scientists** | Fine-tune models, run evaluations, manage deployments | [Fine-tuning](https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/fine-tuning-considerations) |
| **IT admins & platform engineers** | Govern AI resources, enforce policies, manage access | [Control Plane](https://learn.microsoft.com/en-us/azure/foundry/control-plane/overview) |

---
## SDKs and Developer Surfaces

### Unified SDK: `azure-ai-projects` 2.x

The new SDK replaces multiple previous packages:

| Old (Multiple Packages) | New (Unified) |
|---|---|
| `azure-ai-inference` | `azure-ai-projects` 2.x |
| `azure-ai-generative` | `azure-ai-projects` 2.x |
| `azure-ai-ml` | `azure-ai-projects` 2.x |
| `AzureOpenAI()` client | `AIProjectClient.get_openai_client()` → standard `OpenAI()` |

### Available SDK Languages

| Language | Status |
|---|---|
| **Python** | GA |
| **C#** | GA |
| **JavaScript / TypeScript** | Preview |
| **Java** | Preview |

### Developer Surfaces

| Surface | Description |
|---|---|
| **Foundry Portal** | [ai.azure.com](https://ai.azure.com) — manage projects, deploy models, build agents, monitor assets |
| **VS Code Extension** | Explore models and develop agents directly in your IDE |
| **REST API** | Stable v1 routes at `/openai/v1/` |
| **CLI** | Azure CLI integration for resource management |

---
## Hands-On: Explore the AIProjectClient

In [ ]:
# Explore what methods the AIProjectClient provides
from azure.ai.projects import AIProjectClient

# List the public methods available on AIProjectClient
public_methods = [m for m in dir(AIProjectClient) if not m.startswith("_")]
print("AIProjectClient public methods and attributes:")
print("-" * 50)
for method in public_methods:
    print(f"  • {method}")

### Try different models

Once connected, you can easily switch between models by changing the `model` parameter:

In [ ]:
# Example: Switching between different deployed models
# Uncomment and run whichever model you have deployed in your Foundry project

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
openai = project.get_openai_client()

# List of models to try (use only models you have deployed)
models_to_try = [
    "gpt-4.1-mini",
    # "gpt-4.1",
    # "gpt-4o",
    # "gpt-5-mini",
]

prompt = "Explain quantum computing in one sentence."

for model_name in models_to_try:
    try:
        response = openai.responses.create(
            model=model_name,
            input=prompt,
        )
        print(f"[{model_name}]: {response.output_text}")
        print()
    except Exception as e:
        print(f"[{model_name}]: Error — {e}")
        print()

---
## What's New in Foundry

Recent additions to the platform:

| Feature | Description |
|---|---|
| **Prompt Optimizer** (preview) | Automatically improve agent prompts based on evaluation results |
| **Task Adherence guardrails** (preview) | Keep agentic workflows on track with built-in adherence controls |
| **LangChain & LangGraph integration** | Build and trace agents with popular open-source frameworks |
| **Fireworks model import** (preview) | Bring custom models into Foundry through Fireworks |

> See [What's new in Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/whats-new-foundry) for the full list.

---
## Pricing

- **The platform itself is free** to use and explore
- Pricing occurs at the **deployment level** — each product (models, agents, tools) has its own billing model
- Underlying Azure services (compute, storage, networking) also incur costs

> Use the [Total Economic Impact calculator](https://aka.ms/Foundry-ROI-Calculator) to estimate your ROI.

---
## Summary

| Concept | Key Takeaway |
|---|---|
| **Microsoft Foundry** | Unified Azure PaaS for enterprise AI — agents, models, and tools under one roof |
| **Resource model** | Single Foundry resource with projects (replaces Hub + Azure OpenAI + AI Services) |
| **SDK** | `azure-ai-projects` 2.x — one package, one endpoint, one client |
| **API** | Responses API (`/openai/v1/responses`) — agent-native, stable routes |
| **Models** | 1,900+ models from Microsoft, OpenAI, Anthropic, Meta, Mistral, and more |
| **Agents** | Multi-agent orchestration, 1,400+ tools, memory, Foundry IQ knowledge |
| **Governance** | Unified RBAC, networking, policies, real-time observability |

### What's Next?

In **Notebook 3**, we'll build on this foundation:
- Creating and configuring agents with the Responses API
- Adding tools and memory to agents
- Deploying to Azure App Service

### Useful Links

- [Quickstart: Your first API call](https://learn.microsoft.com/en-us/azure/foundry/quickstarts/get-started-code)
- [Agent Service overview](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/workflow)
- [Foundry Models overview](https://learn.microsoft.com/en-us/azure/foundry/concepts/foundry-models-overview)
- [SDK overview](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/sdk-overview)
- [Foundry portal](https://ai.azure.com)
- [Navigate from classic to new](https://learn.microsoft.com/en-us/azure/foundry/how-to/navigate-from-classic)